# PET preclinal data analysis

**Date: 2026/10/02**

## Student information

| Pairs of students    | Last name | First name |
| -------------------- | ----------| ---------- |
| Student #1           |           |            |
| Student #2           |           |            |

---

## Practical session report

* Please complete the cell lines indicated with `# fill-in here` and include your results, figures, and interpretations directly in this notebook whenever requested.
* Send the completed notebook back to Albertine Dubois - albertine.dubois@cea.fr


# Dynamic PET kinetic modeling example
----

This notebook is widely inspired from Samuel Kuttner (2024-09-24) - samuel.kuttner@uit.no<br>
*Kuttner, S., Luppino, L. T., Convert, L., Sarrhini, O., Lecomte, R., Kampffmeyer, M. C., Sundset, R., & Jenssen, R. (2024). Deep learning derived input function in dynamic [18F]FDG PET imaging of mice. Frontiers in Nuclear Medicine, 4. https://doi.org/10.3389/fnume.2024.1372379*

## Background

Positron emission tomography (PET) is a medical imaging technique for sampling the three dimensional spatial and temporal distribution of an intravenously injected radioactive tracer in a patient. It is an excellent tool for visualizing metabolic, physiological and functional information in cardiology, oncology and neurological disorders, e.g. Alzheimer’s and Parkinson’s disease. The most commonly used tracer is [18F]-Fluorodeoxyglucose (FDG), a glucose analogue that is irreversibly trapped in cells with glucose metabolism.

There are two basic ways to acquire PET data: **Frozen in time** (static), or as a **time sequence** (dynamic). Static PET images are simple to interpret and have a low noise level. However, they show merely a snapshot in time. In contrast, dynamic PET images, although requiring longer scans and having higher noise levels, comprise the whole radiotracer uptake pattern over time. Visual observation of the time frames is insufficient to fully exploit the potential with dynamic PET. Instead, tracer kinetic modeling may be used to assess biological parameters, such as the metabolic rate of glucose, drug receptor occupancy, myocardial blood flow, and oxygen consumption. Such measures are not possible to derive from a static PET image.
Tissue compartment modeling is used to perform quantification in PET imaging. It requires at least one tissue curve and and an arterial input function (AIF).

The physiological system of dynamic processes in the tissue of interest is decomposed into a number of compartments, regarded as chemical species in a physical place, which interact with each other.

In compartmental model:

* injected isotope exists in the tissue in a fixed number of physical or chemical states with specified interconnections among them; the arrows indicate the possible pathways the tracer can follow
* compartmental models can be described in terms of a set of linear, first-order, constant-coefficient, ordinary differential equations (ODE)

Change of tracer concentration in one of the tissue compartments is a linear function of the concentrations in all other compartments

$$\frac{dC_i(t)}{dt}=f_i(C_0(t),C_1(t),C_2(t),...)$$

## Aim
The purpose of this exercise is to learn how to perform tissue compartment modeling in Python using various tissue compartment models.

## Dataset
The data for this exercise is available in this repository under `./Example_data/`.
  
The data consists of two files:
* The AIF of a mouse measured during 45 minutes with 1s sampling interval.
* Tissue curves from four tissue regions: Brain, Left Ventricle, Liver and Myocardium.

<center>
<img src="./assets/Amgen_PET.jpg" width=250>
<img src="./assets/TACs.png" width=500>
</center>

## Step1: Load the dependencies

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd
from scipy.optimize import curve_fit
import os
import warnings
warnings.filterwarnings("ignore")
from AIF_curve_fit import AIF_fit_weights, AIF_fit_three_exp

## Step 2:  Read and plot the AIF data

In dynamic PET imaging, the arterial input function (AIF) represents the tracer concentration in blood as a function of time.<br>
Hence, it describes how the tracer concentration changes in blood over time after injection which plays a central role in kinetic modeling and quantitative PET analysis.

- `AIF_A` is the arterial input function in SUV, delay corrected (shifted) to match the PET data
- `AIF_t` is the time vector in minutes

In [ ]:
# ============================================================
# Read AIF file exported as tab-separated values (.txt)
# ============================================================

PATH_AIF = "./Example_data/AIF_shift_SUV_M3.txt"

df = pd.read_csv(PATH_AIF, sep="\t")

# ============================================================
# Display all column names in the dataframe
# ============================================================

COLS = df.columns
print("Column names:")
print([c for c in COLS], "\n")

# ============================================================
# Extract time vector in minutes
# ============================================================

AIF_t = df["Time [min]"].values

print("AIF_t (time vector):")
print(AIF_t, "\n")

# ============================================================
# Extract shifted AIF activity values in SUV units
# ============================================================

AIF_A = ... # fill-in here

print("AIF_A (activity values):")
print(AIF_A, "\n")

In [ ]:
# ============================================================
# Plot measured AIF
# ============================================================

fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)

ax.plot(..., ..., "o", label="Measured AIF", markersize=4)  # fill-in here

ax.set_xlabel("...") # fill-in here
ax.set_ylabel("...") # fill-in here
ax.set_title("Measured AIF") 
ax.legend(loc="upper right")

plt.show()

## Step 3: Fit the Arterial Input Function (AIF)
### Why Fit the Arterial Input Function?

The measured AIF is often noisy due to limited temporal resolution, counting statistics, and image reconstruction artifacts. Fitting the AIF with an analytical model provides several advantages:

- **Noise reduction:**  
  The fitted curve smooths fluctuations in the measured data while preserving the overall physiological behavior.

- **Continuous representation:**  
  Instead of discrete sampled points, the fitted model provides a continuous function that can be evaluated at any time point.

- **Improved kinetic modeling:**  
  Many compartmental and parametric models require a smooth and stable input function for reliable parameter estimation.

- **Robust estimation of physiological parameters:**  
  A fitted AIF improves the estimation of quantities such as tracer delivery, metabolism, and binding rates.

- **Better characterization of peak and tail behavior:**  
  Analytical models such as the Feng model or multi-exponential models can accurately represent both the rapid bolus peak and the slower washout phase.

In this notebook, the measured AIF is fitted using **two different analytical models** in order to obtain a smooth and physiologically meaningful representation of the tracer kinetics.

### 3.1. Three-Exponential model for AIF Fitting

This model approximates the arterial input function using a sum of three exponential decay components:

$$
AIF(t) =
A_1 e^{-\lambda_1 t}
+
A_2 e^{-\lambda_2 t}
+
A_3 e^{-\lambda_3 t}
$$

where:
- $A_i$ are amplitude coefficients,
- $\lambda_i$ are decay constants.

This simplified model focuses on the washout behavior of the tracer after the peak and is particularly useful for:
- smoothing noisy TACs,
- obtaining compact parametric representations,
- and approximating tracer clearance kinetics.

In [ ]:
# ============================================================
# Fit the measured AIF using the three-exponential model
# ============================================================

three_fit, three_params = AIF_fit_three_exp(AIF_t, AIF_A)

# ============================================================
# Plot measured and fitted AIF
# ============================================================

fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)

# ============================================================
# Main plot
# ============================================================

ax.plot(..., ..., "o", label="Measured AIF", markersize=4)  # fill-in here
ax.plot(..., ..., "-", linewidth=2, label="Fitted AIF")  # fill-in here

ax.set_xlabel("Time [min]")
ax.set_ylabel("Activity SUV [g/ml]")
ax.set_title("Three-Exponential AIF Fit")
ax.legend(loc="upper right")

# ============================================================
# Adjust y-axis limits
# ============================================================

ax.set_ylim(top=10.5)

# ============================================================
# Inset axis for early peak visualization
# ============================================================

axins = ax.inset_axes([0.45, 0.45, 0.5, 0.4])

axins.plot(AIF_t, AIF_A, "o", markersize=3)
axins.plot(AIF_t, three_fit, "-", linewidth=2)

axins.set_xlim(0,6)
axins.set_ylim(0,10)

axins.set_xlabel("Time [min]", fontsize=8)
axins.set_ylabel("SUV", fontsize=8)

# ============================================================
# Draw zoom box
# ============================================================

rect, connectors = ax.indicate_inset_zoom(axins)

for c in connectors:
    c.set_color("gray")
    c.set_linewidth(1.5)
    c.set_clip_on(False)

plt.show()

As you can observe, using only exponential decay functions, this approach mainly models the post-peak kinetics. <br>
As an alternative to the three-exponential model, the arterial input function can also be approximated using the Feng-1993 model, which explicitly describes both the bolus arrival and clearance phases.

### 3.2. Feng Model for AIF Fitting

The Feng model is a widely used analytical model for describing the arterial input function (AIF) in dynamic PET imaging. It represents the tracer concentration in blood over time using a combination of exponential terms and a delay parameter.  

This model is particularly useful because it can accurately capture both:

- the rapid bolus peak immediately after tracer injection,
- and the slower washout phase observed at later times.

Mathematically, the Feng model is defined as:

$$
C_p(t) =
\left(A_1 (t-\tau) - A_2 - A_3\right)e^{L_1(t-\tau)}
+ A_2 e^{L_2(t-\tau)}
+ A_3 e^{L_3(t-\tau)}
$$

for $t > \tau$, and $C_p(t) = 0 \quad \text{for} \quad t < \tau$

where:

- $\tau$ is the arrival delay time,
- $A_1, A_2, A_3$ are amplitude coefficients,
- $L_1, L_2, L_3$ are exponential decay constants.

In this notebook, the Feng model is fitted to the measured AIF using nonlinear least-squares optimization. The fitted curve is then compared to the measured data to evaluate the quality of the approximation.

In [ ]:
# ============================================================
# Fit the measured AIF using the Feng model
# ============================================================

feng_fit, feng_params = AIF_fit_weights(AIF_t, AIF_A)

# ============================================================
# Plot measured and fitted AIF
# ============================================================

fig, ax = plt.subplots(figsize=(10, 6), constrained_layout=True)

# ============================================================
# Main plot
# ============================================================

ax.plot(..., ..., "o", label="Measured AIF", markersize=4) # fill-in here
ax.plot(..., ..., "-", linewidth=2, label="Fitted AIF") # fill-in here

ax.set_xlabel("Time [min]")
ax.set_ylabel("Activity SUV [g/ml]")
ax.set_title("Feng AIF Fit")
ax.legend(loc="upper right")

# ============================================================
# Adjust y-axis limits
# ============================================================

ax.set_ylim(top=10.5)

# ============================================================
# Inset axis for early peak visualization
# ============================================================

axins = ax.inset_axes([0.45, 0.45, 0.5, 0.4])

axins.plot(AIF_t, AIF_A, "o", markersize=3)
axins.plot(AIF_t, feng_fit, "-", linewidth=2)

axins.set_xlim(0,6)
axins.set_ylim(0,10)

axins.set_xlabel("Time [min]", fontsize=8)
axins.set_ylabel("SUV", fontsize=8)

# ============================================================
# Draw zoom box
# ============================================================

rect, connectors = ax.indicate_inset_zoom(axins)

for c in connectors:
    c.set_color("gray")
    c.set_linewidth(1.5)
    c.set_clip_on(False)

plt.show()

-----
## Evaluation
Compare both AIF fitting methods and conclude ....

----

## Step 4: Kinetic modelling

Once fitted, AIF is used as the input function in compartmental models.

The simplest compartmental model is the **two-compartmental model**, or **one-tissue compartmental model** (1TCM). First compartment is for the input function (concentration of the tracer in the blood or plasma pool), which we have measured, and therefore we do not need an equation for that. The seconds compartment is for the isotope label in tissue. Compartments are connected with two rate constants, $K_1$ and $k_2$. 

<center>
<img src="./assets/1tcm.png" width=250>
</center>

With **two tissue compartments** (2TCM), we need to have two ordinary differential equations to calculate the concentration curves in both of them:
For example, the kinetic model for measurement of glucose transport and phosphorylation rate in the brain using FDG is based on a three-compartment model. If tracer uptake is irreversible, we can set $k_4=0$, which simplifies the differential equations.

<center>
<img src="./assets/2tcm.png" width=450>
</center>


### 4.1. Read and plot the VOI data

In [ ]:
# ==========================================================
# Load PMOD VOI TAC data
# ==========================================================

PATH_VOI = "./Example_data/VOIdata_M3.voistat"

# ==========================================================
# Read the PMOD VOI file
# header=7 skips the metadata lines at the top of the file
# ==========================================================

df = pd.read_csv(PATH_VOI, sep="\t", header=7)

# ==========================================================
# Extract VOI names
# ==========================================================

# Get the VOI names for each row
VOI_names =df['VoiName(Region) [string]']

# Extract unique VOI names and their first occurrence index
VOI_names_unique, VOI_start_idx = np.unique(VOI_names, return_index=True)

print("Detected VOIs:")
print(VOI_names_unique)

# ==========================================================
# Determine number of dynamic frames
# ==========================================================

# Total number of time frames per VOI
num_frames = ... # fill-in here

print("\nNumber of frames:", num_frames)

# ==========================================================
# Store TACs in a dictionary
# ==========================================================

VOI_A_dict = {}

for idx, start_idx in enumerate(VOI_start_idx):

    # Start/end rows corresponding to one VOI
    start = start_idx
    end = start_idx + num_frames - 1

    # Extract TAC activity values (SUV)
    VOI_A = df.loc[start:end, 'Averaged [SUV]'].values

    # Store TAC using VOI name as key
    VOI_A_dict.update({
        VOI_names_unique[idx]: VOI_A
    })

# ==========================================================
# Extract frame timing information
# ==========================================================

# Time values are identical for all VOIs
# Use the last extracted VOI indices
VOI_t = df.loc[start:end, 'Time [seconds]'].values

# ==========================================================
# Convert PMOD frame center times to frame start times
# ==========================================================

VOI_t_shift = np.zeros_like(VOI_t)
for idx, time in enumerate(VOI_t):
    if idx == 0:
        VOI_t_shift[idx] = 0
    else:
        VOI_t_shift[idx] = 2*VOI_t[idx-1] - VOI_t_shift[idx-1]

# Convert seconds → minutes
VOI_t_shift = ... # fill-in here

# ==========================================================
# Plot TACs
# ==========================================================

plt.figure(figsize=(10,6))

plt.plot(
    VOI_t_shift,
    VOI_A_dict[VOI_names_unique[0]],
    label=VOI_names_unique[0]
)

plt.plot(
    ... # fill-in here
)

plt.plot(
    ... # fill-in here
)

plt.plot(
    ... # fill-in here
)

plt.xlabel("Time [min]")
plt.ylabel("Activity SUV [g/ml]")
plt.title("VOI Time-Activity Curves")
plt.legend()

plt.show()

### 4.2 Apply tissue compartment models to VOI data

Now it is time to define the functions for tissue compartment modeling (TCM). We will evaluate three different models, one-tissue compartment model, and reversible and irreversible two-tissue compartment model.

Thus, we need three functions:
1. An interpolation function that interpolates the AIF and VOI data to the same and constant time sampling, as well as shortens the longest to obtain equal length.

2. A function where the equations for the different models are defined.

3. An optimization function, using the `curve_fit` from the SciPy package.

In [ ]:
from Tissue_compartment_modeling import (
    interpolate_time_frames,
    patlak,
    twoTCMirrev,
    twoTCMrev,
    oneTCM
)

Perform TCM of all tissues in the `*.voistat` file using either unfitted or fitted AIF.

In [ ]:
AIF_A = AIF_A.copy() # unfitted AIF
AIF_A = ... # fill-in here to replace with Feng or three-exponential fitted AIF

In [ ]:
# ==========================================================
# Initialize dictionaries for kinetic modeling results
# Each TCM_dict will store all the resuts from kinetic modeling (k-values)
# ==========================================================

# One-tissue compartment model (fixed vB)
oneTCM_dict = {}

# One-tissue compartment model (fitted vB)
oneTCM_fittedvB_dict = {}

# Reversible two-tissue compartment model
twoTCMrev_dict = {}

# Irreversible two-tissue compartment model
twoTCMirrev_dict = {}

# Patlak graphical analysis
patlak_dict = {}

# ==========================================================
# Loop over all VOIs / tissues
# ==========================================================

for tissue in VOI_names_unique:
    print("Processing tissue:", tissue)

    # Measured TAC for current tissue
    tissue_TAC = VOI_A_dict[tissue]

    # ======================================================
    # 1TCM with fitted blood volume fraction (vB)
    # ======================================================

    (K1, k2, vB), tissue_fit, mse = oneTCM(
        AIF_A,
        AIF_t,
        tissue_TAC,
        VOI_t_shift,
        INTERPOLATE=True,
        TIME_FRAMES=2.5
    )

    # Store fitted parameters
    oneTCM_fittedvB_dict[tissue] = {
        "K1": K1,
        "k2": k2,
        "vB": vB,
        "tissue_fit": tissue_fit,
        "mse": mse
    }

    # ======================================================
    # 1TCM with fixed blood volume fraction
    # ======================================================

    vB = 0.02

    (K1, k2), tissue_fit, mse = oneTCM(
        AIF_A,
        AIF_t,
        tissue_TAC,
        VOI_t_shift,
        INTERPOLATE=True,
        TIME_FRAMES=2.5,
        fit_vB=False,
        vB_fixed=vB
    )

    # Store fitted parameters
    oneTCM_dict[tissue] = {
        "K1": K1,
        "k2": k2,
        "vB": vB,
        "tissue_fit": tissue_fit,
        "mse": mse
    }

    # ======================================================
    # Reversible 2TCM
    # ======================================================

    # Fixed blood volume fraction
    vB = 0.02

    (K1, k2, k3, k4), Ki, tissue_fit, mse = twoTCMrev(
        AIF_A,
        AIF_t,
        tissue_TAC,
        VOI_t_shift,
        INTERPOLATE=True,
        TIME_FRAMES=2.5,
        fit_vB=False,
        vB_fixed=vB
    )

    # Store fitted parameters
    twoTCMrev_dict[tissue] = {
        "K1": K1,
        "k2": k2,
        "k3": k3,
        "k4": k4,
        "vB": vB,
        "Ki": Ki,
        "tissue_fit": tissue_fit,
        "mse": mse
    }

    # ======================================================
    # Irreversible 2TCM
    # ======================================================

    (K1, k2, k3), Ki, tissue_fit, mse = twoTCMirrev(
        AIF_A,
        AIF_t,
        tissue_TAC,
        VOI_t_shift,
        INTERPOLATE=True,
        TIME_FRAMES=2.5,
        fit_vB=False,
        vB_fixed=0.02
    )

    # Store fitted parameters
    twoTCMirrev_dict[tissue] = {
        "K1": K1,
        "k2": k2,
        "k3": k3,
        "vB": vB,
        "Ki": Ki,
        "tissue_fit": tissue_fit,
        "mse": mse
    }

    # ======================================================
    # Patlak graphical analysis
    # ======================================================
    
    Ki, V0 = patlak(
        AIF_A,
        AIF_t,
        tissue_TAC,
        VOI_t_shift,
        INTERPOLATE=True,
        TIME_FRAMES=2.5 / 60
    )

    # Store Patlak parameters
    patlak_dict[tissue] = {
        "Ki": Ki,
        "V0": V0,
    }

# ==========================================================
# End of kinetic modeling
# ==========================================================


Print the results

In [ ]:
# ==========================================================
# Display all fitted model dictionaries
# ==========================================================

print("===== 1TCM with fitted vB =====")
print(oneTCM_fittedvB_dict)

print("\n===== 1TCM with fixed vB =====")
print(oneTCM_dict)

print("\n===== Irreversible 2TCM =====")
print(twoTCMirrev_dict)

print("\n===== Reversible 2TCM =====")
print(twoTCMrev_dict)

print("\n===== Patlak analysis =====")
print(patlak_dict)

# ==========================================================
# Print Ki values for each tissue
# ==========================================================

print("\n========================================")
print("Comparison of Ki values")
print("========================================")

for tissue in VOI_names_unique:

    print(f"\nTissue: {tissue}")

    print(
        "Patlak Ki      :",
        round(patlak_dict[tissue]["Ki"], 3)
    )

    print(
        "2TCM reversible Ki :",
        round(twoTCMrev_dict[tissue]["Ki"], 3)
    )

    print(
        "2TCM irreversible Ki :",
        round(twoTCMirrev_dict[tissue]["Ki"], 3)
    )

Plot the resulting tissue curves

In [ ]:
# ==========================================================
# Plot tissue TACs and compartment model fits
# ==========================================================

# List of tissues to display
tissues_to_plot = ["Brain", "Myocardium", "LV", "Liver"]

for tissue in tissues_to_plot:

    plt.figure(figsize=(10,6))

    # ------------------------------------------------------
    # Measured tissue TAC
    # ------------------------------------------------------

    plt.plot(
        VOI_t_shift,
        VOI_A_dict[tissue],
        "ko-",
        label="Measured TAC"
    )

    # ------------------------------------------------------
    # 1TCM fit
    # ------------------------------------------------------

    plt.plot(
        AIF_t,
        oneTCM_dict[tissue]["tissue_fit"],
        label="1TCM fit"
    )

    # ------------------------------------------------------
    # Reversible 2TCM fit
    # ------------------------------------------------------

    plt.plot(
        AIF_t,
        twoTCMrev_dict[tissue]["tissue_fit"],
        label="2TCM reversible fit"
    )

    # ------------------------------------------------------
    # Irreversible 2TCM fit
    # ------------------------------------------------------

    plt.plot(
        AIF_t,
        twoTCMirrev_dict[tissue]["tissue_fit"],
        label="2TCM irreversible fit"
    )

    # ------------------------------------------------------
    # Title with MSE values
    # ------------------------------------------------------

    plt.title(
        f"{tissue}\n"
        f"MSE 1TCM = {oneTCM_dict[tissue]['mse']:.4f} | "
        f"MSE 2TCMrev = {twoTCMrev_dict[tissue]['mse']:.4f} | "
        f"MSE 2TCMirrev = {twoTCMirrev_dict[tissue]['mse']:.4f}"
    )

    # ------------------------------------------------------
    # Labels and legend
    # ------------------------------------------------------

    plt.xlabel("Time [min]")
    plt.ylabel("Activity [SUV]")
    plt.legend()

    plt.show()

-----
## Evaluation

* What is MSE?
* What is it used for?
* Compare results obtained with each compartmental model method and conclude ...
  
-----